In [ ]:
import os
import glob
import time
import pandas as pd
from tqdm import tqdm
from metrics import BehaviorMatch

data_dir = 'exp/attack'

In [ ]:
def evaluate_file(llm_judge, response_file):
    print(response_file)
    response_df = pd.read_json(response_file, lines=True)
    for i in tqdm(range(len(response_df))):
        if pd.notnull(response_df.loc[i, 'BMSR']):
            continue
        try:
            response_df.loc[i, 'BMSR'] = llm_judge.check_success(
                response_df.loc[i, 'behavior'], 
                response_df.loc[i, 'adv_resp']
            )
        except Exception as e:
            print(f"Error processing index {i}: {e}")
            response_df.loc[i, 'BMSR'] = None
    response_df.to_json(response_file, orient='records', lines=True, force_ascii=False)
    bmsr = response_df.groupby('behavior')['BMSR'].mean()
    print(bmsr)
    time.sleep(3)

In [ ]:
judge_name = 'qwen'
llm_judge = BehaviorMatch(judge_name)

response_files = sorted(glob.glob(os.path.join(data_dir, '*', '*', '*.jsonl')))
for response_file in response_files:
    evaluate_file(llm_judge, response_file)

In [ ]:
judge_name = 'gpt'
llm_judge = BehaviorMatch(judge_name)

response_files = sorted(glob.glob(os.path.join(data_dir, '*', '*', '*.jsonl')))
for response_file in response_files:
    evaluate_file(llm_judge, response_file)

In [ ]:
judge_name = 'gemini'
llm_judge = BehaviorMatch(judge_name)

response_files = sorted(glob.glob(os.path.join(data_dir, '*', '*', '*.jsonl')))
for response_file in response_files:
    evaluate_file(llm_judge, response_file)